This notebook serves to add the docking scores calculated using MOE for each batch of molecules to their respective batch CSV files.

In [1]:
# Import necessary packages
import pandas as pd
import sys
from pathlib import Path

# Import add_docking_scores function
project_root = Path.cwd().parents[0]  # goes up two levels: batches -> data -> cmse802_project
sys.path.append(str(project_root))
from src.Add_Docking_Scores import add_docking_scores

In [3]:
# Initialize a dictionary to store intermediate dataframes
dfs = {}

# Modify index range for each batch
for j in range(1,5):
    # Read the input CSV file
    df = pd.read_csv(f"../data/batches/batch_{j}_scores.csv")
    for i in range(len(df["Index"])):
        # Update indices to range 0-4000
        df.loc[i, "Index"] = df.loc[i, "Index"] + (1000 * (j - 1))
    # Add updated dataframe to dictionary
    dfs[j-1] = df

# Concatenate batches into a single dataframe
df_combined = pd.concat([dfs[j] for j in range(4)])
df_combined.to_csv("../data/batches/combined_scores.csv", index=False)

# Add docking scores to the dataframe
df_with_scores = add_docking_scores("../data/data_with_tanimoto.csv", 
                                    "../data/batches/combined_scores.csv", 
                                    "../data/data_with_scores.csv", 
                                    drop_na=False)

# Remove rows with no docking score and save the final dataframe to a CSV file
df_final = df_with_scores.dropna(subset=["Docking Score"])
df_final.to_csv("../data/data_final.csv", index=False)

# Save the removed rows to a separate CSV file
df_removed = df_with_scores[df_with_scores["Docking Score"].isna()]
df_removed.to_csv("../data/data_no_scores.csv", index=False)